In [1]:
# ==========================================
# Find Spark
# ==========================================
import findspark
findspark.init()

In [3]:
# ==========================================
# Imports
# ==========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [4]:
# ==========================================
# Create Spark Session
# ==========================================

spark = SparkSession.builder \
    .appName("API Data Pipeline") \
    .getOrCreate()

In [5]:
# ==========================================
# Read Bronze Layer
# ==========================================

df = spark.read.option("multiline", "true") \
               .json("D:/api-data-pipeline/data/raw/posts.json")
df.show(5, truncate=False)
df.printSchema()

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+--------------------------------------------------------------------------+------+
|body                                                                                                                                                                                                             |id |title                                                                     |userId|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+--------------------------------------------------------------------------+------+
|quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas to

In [6]:
##Title Lenght Function
from pyspark.sql import functions as F

df = df.withColumn(
    "title_length",
    F.length("title")
)

df.select(
    "id",
    "title",
    "title_length"
).show(10, truncate=False)

+---+--------------------------------------------------------------------------+------------+
|id |title                                                                     |title_length|
+---+--------------------------------------------------------------------------+------------+
|1  |sunt aut facere repellat provident occaecati excepturi optio reprehenderit|74          |
|2  |qui est esse                                                              |12          |
|3  |ea molestias quasi exercitationem repellat qui ipsa sit aut               |59          |
|4  |eum et est occaecati                                                      |20          |
|5  |nesciunt quas odio                                                        |18          |
|6  |dolorem eum magni eos aperiam quia                                        |34          |
|7  |magnam facilis autem                                                      |20          |
|8  |dolorem dolore est ipsam                               

In [7]:
# ==========================================
# Writing to Silver Layer
# ==========================================
df.write \
  .mode("overwrite") \
  .parquet("D:/api-data-pipeline/data/silver/posts")

In [8]:
# ==========================================
# Utility Functions
# ==========================================

from pyspark.sql import functions as F
def profile_data(df):
    print("=" * 60)
    print("DATA PROFILING REPORT")
    print("=" * 60)

    print(f"Total Rows    : {df.count()}")
    print(f"Total Columns : {len(df.columns)}")

    print("\nColumns:")
    print(df.columns)

    print("\nSchema:")
    df.printSchema()


def null_report(df):

    df.select(
        [
            F.count(
                F.when(F.col(column).isNull(), column)
            ).alias(column)

            for column in df.columns
        ]
    ).show()

def duplicate_report(df):
    total_records = df.count()
    unique_records = df.dropDuplicates().count()
    duplicate_records = total_records - unique_records 
    print("=" * 60)
    print("DUPLICATE REPORT")
    print("=" * 60)

    print(f"Total Records      : {total_records}")
    print(f"Unique Records     : {unique_records}")
    print(f"Duplicate Records  : {duplicate_records}")

def data_type_report(df):
    print("=" * 60)
    print("DATA TYPE REPORT")
    print("=" * 60)

    for column, datatype in df.dtypes:
        print(f"{column:<20} {datatype}")

        
def summary_statistics(df):

    print("=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)

    df.describe().show()

    

    

    

In [9]:
# ==========================================
# Execute
# ==========================================

profile_data(df)
null_report(df)
duplicate_report(df)
data_type_report(df)
summary_statistics(df)

DATA PROFILING REPORT
Total Rows    : 100
Total Columns : 5

Columns:
['body', 'id', 'title', 'userId', 'title_length']

Schema:
root
 |-- body: string (nullable = true)
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- userId: long (nullable = true)
 |-- title_length: integer (nullable = true)

+----+---+-----+------+------------+
|body| id|title|userId|title_length|
+----+---+-----+------+------------+
|   0|  0|    0|     0|           0|
+----+---+-----+------+------------+

DUPLICATE REPORT
Total Records      : 100
Unique Records     : 100
Duplicate Records  : 0
DATA TYPE REPORT
body                 string
id                   bigint
title                string
userId               bigint
title_length         int
SUMMARY STATISTICS
+-------+--------------------+------------------+--------------------+------------------+------------------+
|summary|                body|                id|               title|            userId|      title_length|
+-------+---

In [10]:
new_df = df.withColumn(
    "title_category",
    F.when(
        F.col("title_length") <= 20,
        F.lit("Short")
    ).otherwise(
        F.lit("Long")
    )
)

new_df.select(
    "id",
    "title_length",
    "title_category"
).show(20)


+---+------------+--------------+
| id|title_length|title_category|
+---+------------+--------------+
|  1|          74|          Long|
|  2|          12|         Short|
|  3|          59|          Long|
|  4|          20|         Short|
|  5|          18|         Short|
|  6|          34|          Long|
|  7|          20|         Short|
|  8|          24|          Long|
|  9|          50|          Long|
| 10|          27|          Long|
| 11|          32|          Long|
| 12|          37|          Long|
| 13|          50|          Long|
| 14|          25|          Long|
| 15|          23|          Long|
| 16|          67|          Long|
| 17|          49|          Long|
| 18|          42|          Long|
| 19|          41|          Long|
| 20|          34|          Long|
+---+------------+--------------+
only showing top 20 rows



In [11]:
final_df = new_df.select(
    "id",
    "userId",
    "title",
    "title_length",
    "title_category"
)
final_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- userId: long (nullable = true)
 |-- title: string (nullable = true)
 |-- title_length: integer (nullable = true)
 |-- title_category: string (nullable = false)



In [12]:
# ==========================================
# Post Transformation Validations
# ==========================================

profile_data(final_df)
null_report(final_df)
duplicate_report(final_df)
data_type_report(final_df)
summary_statistics(final_df)

DATA PROFILING REPORT
Total Rows    : 100
Total Columns : 5

Columns:
['id', 'userId', 'title', 'title_length', 'title_category']

Schema:
root
 |-- id: long (nullable = true)
 |-- userId: long (nullable = true)
 |-- title: string (nullable = true)
 |-- title_length: integer (nullable = true)
 |-- title_category: string (nullable = false)

+---+------+-----+------------+--------------+
| id|userId|title|title_length|title_category|
+---+------+-----+------------+--------------+
|  0|     0|    0|           0|             0|
+---+------+-----+------------+--------------+

DUPLICATE REPORT
Total Records      : 100
Unique Records     : 100
Duplicate Records  : 0
DATA TYPE REPORT
id                   bigint
userId               bigint
title                string
title_length         int
title_category       string
SUMMARY STATISTICS
+-------+------------------+------------------+--------------------+------------------+--------------+
|summary|                id|            userId|         

In [13]:
# ==========================================
# Writing to Silver Layer Final Table
# ==========================================
final_df.write \
  .mode("overwrite") \
  .parquet("D:/api-data-pipeline/data/silver/posts")

In [14]:
silver_df = spark.read.parquet(
    "D:/api-data-pipeline/data/silver/posts"
)
silver_df.show(5, truncate=False)
print("Silver Record Count:", silver_df.count())

+---+------+--------------------------------------------------------------------------+------------+--------------+
|id |userId|title                                                                     |title_length|title_category|
+---+------+--------------------------------------------------------------------------+------------+--------------+
|1  |1     |sunt aut facere repellat provident occaecati excepturi optio reprehenderit|74          |Long          |
|2  |1     |qui est esse                                                              |12          |Short         |
|3  |1     |ea molestias quasi exercitationem repellat qui ipsa sit aut               |59          |Long          |
|4  |1     |eum et est occaecati                                                      |20          |Short         |
|5  |1     |nesciunt quas odio                                                        |18          |Short         |
+---+------+------------------------------------------------------------